<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Early Stopping</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Early stopping consiste en detener el entrenamiento cuando la validación deja de mejorar de forma útil. En este notebook se presenta como una técnica simple pero muy práctica para controlar sobreajuste.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Evita seguir ajustando la red cuando el aprendizaje adicional ya no mejora la generalización.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A interpretar la lógica de parada temprana y su relación con la monitorización del rendimiento en validación.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Entrenar más tiempo no siempre produce un modelo mejor; a veces solo produce más sobreajuste.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Early stopping usa señales de validación para decidir cuándo conviene detener el proceso de entrenamiento.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas del conjunto de entrenamiento.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en tensores columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo del entrenamiento.
Estos tensores permiten demostrar cómo el early stopping monitoriza el rendimiento del modelo."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas, etiquetas # Muestra los tensores de entrada y las etiquetas del conjunto de entrenamiento.


In [ ]:
# Bloque 2 — Tensores de validación
"""Este bloque crea tensores que representan las entradas y etiquetas del conjunto de validación.
La variable "entradas_val" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en tensores columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas_val" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo de la validación.
Estos tensores permiten evaluar la métrica de validación en cada época."""

entradas_val = torch.tensor([[2.0, 1.0],
                             [4.0, 3.0],
                             [6.0, 5.0]])

etiquetas_val = torch.tensor([1.0, 0.0, 1.0]).unsqueeze(1)

entradas_val, etiquetas_val # Muestra los tensores de entrada y las etiquetas del conjunto de validación.


In [ ]:
# Bloque 3 — Definición de la red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedEarlyStopping()" que representa un modelo de red neuronal simple.
La clase "RedEarlyStopping()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedEarlyStopping()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos el resultado de llamar a "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos el resultado de llamar a "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo aplicando las capas lineales a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedEarlyStopping()", representando el modelo de red neuronal definido.
La estructura permite demostrar cómo el early stopping monitoriza el rendimiento del modelo."""

import torch.nn as nn

class RedEarlyStopping(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedEarlyStopping()

red # Muestra el modelo de red neuronal definido.


In [ ]:
# Bloque 4 — Función de pérdida y optimizador
"""Este bloque define la función de pérdida y el optimizador utilizados durante el entrenamiento.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto calculador de la función de pérdida.

Aplicamos la clase "torch.optim.Adam()" a la variable "optimizador" para crear un objeto optimizador de parámetros de la red.
Pasamos como argumentos los parámetros de la red mediante la expresión "red.parameters()".
El método ".parameters()" devuelve un iterador sobre los parámetros de la red que serán actualizados por el optimizador.
El parámetro "lr=" especifica la tasa de aprendizaje utilizada por el optimizador, en este caso 0.01.
El optimizador Adam actualiza los parámetros de la red utilizando los gradientes calculados."""

criterio = nn.BCEWithLogitsLoss()

optimizador = torch.optim.Adam(red.parameters(), lr=0.01)

criterio, optimizador # Muestra el objeto calculador de la función de pérdida y el optimizador de parámetros de la red.


In [ ]:
# Bloque 5 — Parámetros del early stopping
"""Este bloque define los parámetros necesarios para aplicar early stopping.
La variable "paciencia" indica cuántas épocas consecutivas se permiten sin mejora, en este caso 3.
Asignamos la expresión "float('inf')" a la variable "mejor_perdida" para indicar que inicialmente no hay un valor de pérdida registrado.
El valor "inf" permite que la primera pérdida finita registrada se considere una mejora.
La variable "mejor_perdida" almacena el mejor valor de pérdida de validación registrado.
La variable "contador" controla cuántas épocas han pasado sin mejora, inicialmente 0.
La variable "mejores_pesos" guarda los parámetros correspondientes a la mejor validación."""

from copy import deepcopy
from math import isfinite

paciencia = 3
mejor_perdida = float("inf")
contador = 0
mejores_pesos = None


In [ ]:
# Bloque 6 — Loop de entrenamiento con early stopping
"""Este bloque implementa el loop de entrenamiento con early stopping.
La variable "epocas" define cuántas veces se repite el ciclo completo de entrenamiento; en este caso, como máximo 20 veces.
La variable "perdidas_val" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida obtenidos durante la validación en cada época.

Primero, realizamos el loop de entrenamiento para actualizar los parámetros de la red.
Utilizamos un bucle "for" para iterar sobre el número de épocas definido en la variable "epocas"; es decir, en este caso se repetirá hasta 20 veces.
La expresión "for _ in range(epocas):" inicia el bucle que se repetirá el número de veces definido en "epocas".
En cada iteración se ejecuta el forward pass, se calcula la pérdida, se reinician los gradientes previos, se ejecuta backward para obtener los gradientes 
y se actualizan los parámetros con el optimizador.

A continuación, realizamos una evaluación para medir el rendimiento del modelo en el conjunto de datos de validación.
Utilizamos la palabra clave "with" para crear un contexto en el que el cálculo de gradientes está desactivado.
Aplicamos la clase "torch.no_grad()" para desactivar el cálculo de gradientes durante la validación en este contexto.
El contexto "with torch.no_grad():" desactiva el cálculo de gradientes para evitar el almacenamiento innecesario de información para backpropagation.
Antes de validar se activa el modo evaluación; al comenzar cada época se activa el modo entrenamiento.

En esta validación se calculan las predicciones del modelo para el conjunto de datos de validación y también se calcula la pérdida correspondiente
al conjunto de datos de validación. Además, se almacena el valor de la pérdida en la lista "perdidas_val" para su posterior análisis.

Implementamos el early stopping para detener el entrenamiento cuando la pérdida de validación deja de mejorar durante un número determinado
de épocas consecutivas; este número se define en la variable "paciencia".

Esta lógica se basa en la comparación de la pérdida de validación actual "perdida_val" con la mejor pérdida registrada hasta el momento
"mejor_perdida" mediante el uso de un bloque condicional "if".

En cada época se calcula la pérdida de entrenamiento y la pérdida de validación. Si la pérdida de validación mejora, se actualiza "mejor_perdida"
y se reinicia el contador. Si no mejora, el contador aumenta para registrar cuántas épocas consecutivas han transcurrido sin mejora.
Cuando el contador alcanza el valor de paciencia, el entrenamiento se detiene mediante "break"; con paciencia 3, ocurre tras 3 épocas sin mejora.
Al final se restauran los parámetros correspondientes a la menor pérdida de validación."""

epocas = 20
perdidas_val = []

for _ in range(epocas): # loop de entrenamiento
    red.train()
    pred_train = red(entradas)
    perdida_train = criterio(pred_train, etiquetas)
    optimizador.zero_grad()
    perdida_train.backward()
    optimizador.step()

    red.eval()
    with torch.no_grad(): # loop de validación
        pred_val = red(entradas_val)
        perdida_val = criterio(pred_val, etiquetas_val)
        perdidas_val.append(float(perdida_val.item()))

    valor_perdida_val = float(perdida_val.item())
    if not isfinite(valor_perdida_val):
        raise ValueError("La pérdida de validación no es finita.")
    if valor_perdida_val < mejor_perdida:
        mejor_perdida = valor_perdida_val
        mejores_pesos = deepcopy(red.state_dict())
        contador = 0
    else:
        contador += 1

    if contador >= paciencia:
        break

if mejores_pesos is None:
    raise RuntimeError("No se registraron pesos de validación.")
red.load_state_dict(mejores_pesos)

# El primer valor de "perdida_val" es el primer valor calculado durante la primera época de validación y 
# también pasa a ser "mejor_perdida" si es finito.

In [ ]:
# Bloque 7 — Resultados finales
"""Este bloque organiza los resultados del proceso de early stopping.
La variable "resultado" almacena en forma de diccionario la mejor pérdida de validación registrada y la secuencia de pérdidas por época.
La estructura permite visualizar las pérdidas y comprobar si el entrenamiento se detuvo antes del máximo de épocas."""

resultado = {
    "mejor_perdida_validacion": mejor_perdida,
    "perdidas_validacion_por_epoca": perdidas_val
}

resultado # Muestra los resultados finales del proceso de early stopping.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo usar la validación para decidir cuándo dejar de entrenar antes de degradar la generalización.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Detener a tiempo también es una forma de regularizar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Elegir criterios de parada sin revisar si la señal monitorizada realmente refleja mejora útil del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Optimización Avanzada para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Optimización Avanzada y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">07_Optimizacion_avanzada/01_La_optimizacion_avanzada.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">